In [1]:
import pandas as pd
import numpy as np 
from sklearn.preprocessing import MinMaxScaler 
import warnings
warnings.filterwarnings('ignore')

In [2]:
print('=' * 60)
print('STEP 01 - DATA COLLECTION')
print('=' * 60)

STEP 01 - DATA COLLECTION


In [13]:
# load dataset
RAW_PATH = r"C:\Users\FLAKES\OneDrive\Documents\last tem\Computer system project\store\raw\retail_store_inventory.csv"
df_raw = pd.read_csv(RAW_PATH)

print(f"loaded dataset: {df_raw.shape[0]:,}rows and {df_raw.shape[1]} columns")
print(f"Date range:{df_raw['Date'].min()} to {df_raw['Date'].max}")
print(f"Categories:{df_raw['Category'].unique().tolist()}")
print(f"\nColumn names:\n{df_raw.columns.tolist()}")
print(f"\nFirst 3 rows:\n{df_raw.head(3).to_string(index=False)}")

loaded dataset: 73,100rows and 15 columns
Date range:2022-01-01 to <bound method Series.max of 0        2022-01-01
1        2022-01-01
2        2022-01-01
3        2022-01-01
4        2022-01-01
            ...    
73095    2024-01-01
73096    2024-01-01
73097    2024-01-01
73098    2024-01-01
73099    2024-01-01
Name: Date, Length: 73100, dtype: object>
Categories:['Groceries', 'Toys', 'Electronics', 'Furniture', 'Clothing']

Column names:
['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast', 'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing', 'Seasonality']

First 3 rows:
      Date Store ID Product ID  Category Region  Inventory Level  Units Sold  Units Ordered  Demand Forecast  Price  Discount Weather Condition  Holiday/Promotion  Competitor Pricing Seasonality
2022-01-01     S001      P0001 Groceries  North              231         127             55           135.47  33.50    

In [19]:
#----Filter to Groceries (perishable goods)----

df=df_raw[df_raw['Category']=='Groceries'].copy()
print("\nFiltered to 'Groceries'(perishable goods)")
print(f"Remaining rows: {len(df):,}")
print(f"Unique Products: {df["Product ID"].nunique()}")
print(f"Unique Stores: {df["Store ID"].nunique()}")



Filtered to 'Groceries'(perishable goods)
Remaining rows: 14,611
Unique Products: 20
Unique Stores: 5


In [20]:
print("\n" + "=" * 60)
print("STEP 02 - DATA CLEANING & FEATURE ENGINEERING")
print('=' * 60)


STEP 02 - DATA CLEANING & FEATURE ENGINEERING


In [39]:
# Parse dates & sort 
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Store ID', 'Product ID', 'Date']).reset_index(drop=True)
print('Dates parsed and rows sorted by stored, Product and Date')
print(df.head())

Dates parsed and rows sorted by stored, Product and Date
        Date Store ID Product ID   Category Region  Inventory Level  \
0 2022-01-01     S001      P0001  Groceries  North              231   
1 2022-01-02     S001      P0001  Groceries   West              116   
2 2022-01-04     S001      P0001  Groceries  South               85   
3 2022-01-05     S001      P0001  Groceries  South              238   
4 2022-01-15     S001      P0001  Groceries  North              290   

   Units Sold  Units Ordered  Demand Forecast  Price  ...  Weather_Enc  \
0       127.0             55           135.47  33.50  ...            2   
1        81.0            104            92.94  27.95  ...            1   
2        58.0            193            52.87  77.88  ...            1   
3       147.0             37           150.27  28.46  ...            0   
4       176.0             94           170.06  98.04  ...            2   

  Season_Enc  Region_North  Region_South Region_West  DayOfWeek  DayOfM

In [27]:
# Handles nulls
null_counts = df.isnull().sum()
print(f"\n Null Value check: \n{null_counts[null_counts > 0] if null_counts.sum() > 0 else 'No nulls found - dataset is complete'}")


 Null Value check: 
No nulls found - dataset is complete


In [38]:
#foward-fill within each SKU group (handles any future nulls robustly)

df[['Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast']]=(
    df.groupby(['Store ID','Product ID'])[
        ['Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast']
        ].transform(lambda x:x.ffill().bfill())
)
print('Foward/back fill applied per SKU group')
print(df.head())

Foward/back fill applied per SKU group
        Date Store ID Product ID   Category Region  Inventory Level  \
0 2022-01-01     S001      P0001  Groceries  North              231   
1 2022-01-02     S001      P0001  Groceries   West              116   
2 2022-01-04     S001      P0001  Groceries  South               85   
3 2022-01-05     S001      P0001  Groceries  South              238   
4 2022-01-15     S001      P0001  Groceries  North              290   

   Units Sold  Units Ordered  Demand Forecast  Price  ...  Weather_Enc  \
0       127.0             55           135.47  33.50  ...            2   
1        81.0            104            92.94  27.95  ...            1   
2        58.0            193            52.87  77.88  ...            1   
3       147.0             37           150.27  28.46  ...            0   
4       176.0             94           170.06  98.04  ...            2   

  Season_Enc  Region_North  Region_South Region_West  DayOfWeek  DayOfMonth  \
0         

In [31]:
#handle outliers

numeric_cols= ['Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast']

print('\n Outliers detection:')
for col in numeric_cols:
    Q1= df[col].quantile(0.25)
    Q3= df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5* IQR
    outliers_mask = (df[col] < lower) | (df[col] < upper)
    n_outliers = outliers_mask.sum()

    df[col] = df[col].clip(lower=lower, upper=upper)
    print(f"{col:25s}: {n_outliers:4d} outliers capped to [{lower:1f}, {upper:1f}]")
    


 Outliers detection:
Inventory Level          : 14611 outliers capped to [-172.000000, 724.000000]
Units Sold               : 14479 outliers capped to [-187.500000, 440.500000]
Units Ordered            : 14611 outliers capped to [-71.500000, 292.500000]
Demand Forecast          : 14469 outliers capped to [-180.447500, 442.652500]


In [35]:
# Encode categorical feature

print('\n Encoding categorical features')

#weather condition
weather_map = {'Sunny': 0, 'Cloudy': 1, 'Rainy':2, 'Snowy': 3}
df['Weather_Enc']= df['Weather Condition'].map (weather_map)
print(f'Weather Condition and Weather_Enc{weather_map}')

# Seasonality
season_map = {'Spring': 0, 'Summer': 1, 'Autumn': 2, 'Winter':3}
df['Season_Enc']= df['Seasonality'].map (season_map)
print(f'Seasonality and Season_Enc{season_map}')

#Region
region_enc = pd.get_dummies(df['Region'], prefix = 'Region', drop_first= True).astype(int)
df= pd.concat([df, region_enc], axis=1)
print(f' Region (one_hot) and {region_enc.columns.tolist()}')


 Encoding categorical features
Weather Condition and Weather_Enc{'Sunny': 0, 'Cloudy': 1, 'Rainy': 2, 'Snowy': 3}
Seasonality and Season_Enc{'Spring': 0, 'Summer': 1, 'Autumn': 2, 'Winter': 3}
 Region (one_hot) and ['Region_North', 'Region_South', 'Region_West']


In [37]:
# Calendar / date features
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['DayOfMonth'] = df['Date'].dt.day
df['Month'] = df['Date'].dt.month
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)
print('\n Calendar feature added: DayOfWeek, DayOfMonth, Month, WeekOfYear, IsWeekend')
print(df[['Date', 'DayOfWeek', 'DayOfMonth', 'Month', 'WeekOfYear', 'IsWeekend']].head())


 Calendar feature added: DayOfWeek, DayOfMonth, Month, WeekOfYear, IsWeekend
        Date  DayOfWeek  DayOfMonth  Month  WeekOfYear  IsWeekend
0 2022-01-01          5           1      1          52          1
1 2022-01-02          6           2      1          52          1
2 2022-01-04          1           4      1           1          0
3 2022-01-05          2           5      1           1          0
4 2022-01-15          5          15      1           2          1


In [40]:
# Lag features (temporal sequences for LSTM)
lag_days = [1, 3, 7, 14]
df= df.sort_values(['Store ID', 'Product ID', 'Date'])

for lag in lag_days:
    df[f'Units_Sold_Lag{lag}'] = (
        df.groupby(['Store ID', 'Product ID'])['Units Sold']
        .shift(lag)
    )
    df[f'Inventory_Lag{lag}'] = (
        df.groupby(['Store ID', 'Product ID'])['Inventory Level']
        .shift(lag)
    )

In [41]:
# Rolling statistics (7-day and 14-day window)
df['Rolling_Mean_7d']  = df.groupby(['Store ID', 'Product ID'])['Units Sold'].transform(lambda x: x.rolling(7,  min_periods=1).mean())
df['Rolling_Std_7d']   = df.groupby(['Store ID', 'Product ID'])['Units Sold'].transform(lambda x: x.rolling(7,  min_periods=1).std().fillna(0))
df['Rolling_Mean_14d'] = df.groupby(['Store ID', 'Product ID'])['Units Sold'].transform(lambda x: x.rolling(14, min_periods=1).mean())

print(f" Lag features created  : Units_Sold_Lag1/3/7/14, Inventory_Lag1/3/7/14")
print(f" Rolling stats created : Rolling_Mean_7d, Rolling_Std_7d, Rolling_Mean_14d")

 Lag features created  : Units_Sold_Lag1/3/7/14, Inventory_Lag1/3/7/14
 Rolling stats created : Rolling_Mean_7d, Rolling_Std_7d, Rolling_Mean_14d


In [42]:
# Drop rows with NaN introduced by lags
before = len(df)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Dropped {before - len(df)} rows with NaN from lag windows {len(df):,} rows remain")

Dropped 1400 rows with NaN from lag windows 13,211 rows remain


In [43]:
# MinMax Scaling 
scale_cols = [
    'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast',
    'Price', 'Competitor Pricing', 'Discount',
    'Units_Sold_Lag1', 'Units_Sold_Lag3', 'Units_Sold_Lag7', 'Units_Sold_Lag14',
    'Inventory_Lag1', 'Inventory_Lag3', 'Inventory_Lag7', 'Inventory_Lag14',
    'Rolling_Mean_7d', 'Rolling_Std_7d', 'Rolling_Mean_14d'
]

scaler = MinMaxScaler(feature_range=(0, 1))
df_scaled = df.copy()
df_scaled[scale_cols] = scaler.fit_transform(df[scale_cols])
print(f"\n MinMax scaling applied (0–1) to {len(scale_cols)} numerical features")


 MinMax scaling applied (0–1) to 18 numerical features


In [44]:
#Final cleaned dataset 
feature_cols = [
    'Date', 'Store ID', 'Product ID',
    # Target
    'Units Sold',
    # Inventory & demand
    'Inventory Level', 'Units Ordered', 'Demand Forecast',
    # Price signals
    'Price', 'Discount', 'Competitor Pricing',
    # Encoded categoricals
    'Weather_Enc', 'Season_Enc', 'Holiday/Promotion',
    # Region dummies
] + [c for c in df.columns if c.startswith('Region_')] + [
    # Calendar
    'DayOfWeek', 'DayOfMonth', 'Month', 'WeekOfYear', 'IsWeekend',
    # Lags
    'Units_Sold_Lag1', 'Units_Sold_Lag3', 'Units_Sold_Lag7', 'Units_Sold_Lag14',
    'Inventory_Lag1', 'Inventory_Lag3', 'Inventory_Lag7', 'Inventory_Lag14',
    # Rolling stats
    'Rolling_Mean_7d', 'Rolling_Std_7d', 'Rolling_Mean_14d'
]

df_clean = df_scaled[feature_cols].copy()

In [50]:
#Train / Test split (80/20, time-ordered)
split_idx = int(len(df_clean) * 0.80)
df_train = df_clean.iloc[:split_idx]
df_test  = df_clean.iloc[split_idx:]

print(f"\n Train/Test split (time-ordered, no shuffle):")
print(f" Train : {len(df_train):,} rows  ({df_train['Date'].min().date()} → {df_train['Date'].max().date()})")
print(f" Test  : {len(df_test):,}  rows  ({df_test['Date'].min().date()}  → {df_test['Date'].max().date()})")



 Train/Test split (time-ordered, no shuffle):
 Train : 10,568 rows  (2022-02-11 → 2024-01-01)
 Test  : 2,643  rows  (2022-02-17  → 2024-01-01)


In [47]:
#Save cleaned outputs 
df_clean.to_csv("cleaned_grocery_inventory.csv",    index=False)
df_train.to_csv("train_grocery_inventory.csv",      index=False)
df_test.to_csv( "test_grocery_inventory.csv",       index=False)

print(f"\nSaved files:")
print(f"cleaned_grocery_inventory.csv  — full cleaned dataset")
print(f"train_grocery_inventory.csv    — training set (80%)")
print(f"test_grocery_inventory.csv     — test set (20%)")


Saved files:
  cleaned_grocery_inventory.csv  — full cleaned dataset
  train_grocery_inventory.csv    — training set (80%)
  test_grocery_inventory.csv     — test set (20%)


In [53]:
#Summary report
print("\n" + "=" * 60)
print("STEPS 01 & 02 COMPLETE — SUMMARY")
print("=" * 60)

print(f"  Raw dataset rows       : {len(df_raw):,}")
print(f"  Groceries rows (Step1) : {df_raw[df_raw['Category']=='Groceries'].shape[0]:,}")
print(f"  After cleaning (Step2) : {len(df_clean):,}")
print(f"  Features for model     : {len(feature_cols) - 3}  (excl. Date, Store ID, Product ID)")
print(f"  Target variable        : Units Sold  (scaled 0–1)")
print(f"  Train rows             : {len(df_train):,}")
print(f"  Test rows              : {len(df_test):,}")


STEPS 01 & 02 COMPLETE — SUMMARY
  Raw dataset rows       : 73,100
  Groceries rows (Step1) : 14,611
  After cleaning (Step2) : 13,211
  Features for model     : 29  (excl. Date, Store ID, Product ID)
  Target variable        : Units Sold  (scaled 0–1)
  Train rows             : 10,568
  Test rows              : 2,643
